In [2]:
import cmath
import math
import numpy as np

# ==========================================
# Hardware Quantization Configuration
# ==========================================
D_BITS = 18; D_FRAC = 17  # Data: Q1.17
T_BITS = 20; T_FRAC = 17  # Twiddles: Q3.17

def to_hex(val, bits, frac_bits):
    """
    Converts a float to a Two's Complement Hex string, 
    matching your specific Q-format and bit widths.
    """
    scaled = int(round(val * (1 << frac_bits)))
    
    # Calculate limits for Saturation (prevent wrap-around overflow)
    max_val = (1 << (bits - 1)) - 1
    min_val = -(1 << (bits - 1))
    
    if scaled > max_val:
        scaled = max_val
    elif scaled < min_val:
        scaled = min_val
        
    mask = (1 << bits) - 1
    hex_chars = (bits + 3) // 4
    return f"0x{(scaled & mask):0{hex_chars}X}"

def hex_cpx(c_val, bits, frac_bits):
    return f"{to_hex(c_val.real, bits, frac_bits)} + j {to_hex(c_val.imag, bits, frac_bits)}"

# ==========================================
# FFT Math Core
# ==========================================
def w(n, k):
    return cmath.exp(-2j * cmath.pi * k / n)

def radix4_butterfly(a, b, c, d, w1, w2, w3):
    s0 = a + c; s1 = a - c; s2 = b + d; s3 = b - d
    A_pre = s0 + s2; B_pre = s1 - 1j * s3; C_pre = s0 - s2; D_pre = s1 + 1j * s3
    
    return A_pre, B_pre * w1, C_pre * w2, D_pre * w3

def radix2_butterfly(a, b):
    return a + b, a - b

# ==========================================
# MDC Pipeline Simulation
# ==========================================
def simulate_mdc_pipeline(input_data, verbose=True):
    """
    Simulates the pipeline. 
    Set verbose=False to suppress Hex X-Ray printing during bulk testing.
    """
    stage1_out = [0j] * 128
    if verbose: print("\n" + "="*95 + "\n STAGE 1 (Radix-4)\n" + "="*95)
    for i in range(32):
        a, b, c, d = input_data[i], input_data[i+32], input_data[i+64], input_data[i+96]
        w1, w2, w3 = w(128, i), w(128, 2*i), w(128, 3*i)
        A, B, C, D = radix4_butterfly(a, b, c, d, w1, w2, w3)
        stage1_out[i], stage1_out[i+32], stage1_out[i+64], stage1_out[i+96] = A, B, C, D

        if verbose:
            print(f"[Clk {i:02d}] In:  a={hex_cpx(a, D_BITS, D_FRAC)}, b={hex_cpx(b, D_BITS, D_FRAC)}, c={hex_cpx(c, D_BITS, D_FRAC)}, d={hex_cpx(d, D_BITS, D_FRAC)}")
            print(f"          Tw:  w1={hex_cpx(w1, T_BITS, T_FRAC)}, w2={hex_cpx(w2, T_BITS, T_FRAC)}, w3={hex_cpx(w3, T_BITS, T_FRAC)}")
            print(f"          Out: A={hex_cpx(A/4, D_BITS, D_FRAC)}, B={hex_cpx(B/4, D_BITS, D_FRAC)}, C={hex_cpx(C/4, D_BITS, D_FRAC)}, D={hex_cpx(D/4, D_BITS, D_FRAC)}\n")

    stage2_out = [0j] * 128
    if verbose: print("="*95 + "\n STAGE 2 (Radix-4)\n" + "="*95)
    clk = 0
    for block in range(4): 
        offset = block * 32
        for i in range(8):
            a, b, c, d = stage1_out[offset+i], stage1_out[offset+i+8], stage1_out[offset+i+16], stage1_out[offset+i+24]
            w1, w2, w3 = w(32, i), w(32, 2*i), w(32, 3*i)
            A, B, C, D = radix4_butterfly(a, b, c, d, w1, w2, w3)
            stage2_out[offset+i], stage2_out[offset+i+8], stage2_out[offset+i+16], stage2_out[offset+i+24] = A, B, C, D

            if verbose:
                print(f"[Clk {clk:02d}] Out: A={hex_cpx(A/16, D_BITS, D_FRAC)}, B={hex_cpx(B/16, D_BITS, D_FRAC)}, C={hex_cpx(C/16, D_BITS, D_FRAC)}, D={hex_cpx(D/16, D_BITS, D_FRAC)}")
            clk += 1

    stage3_out = [0j] * 128
    if verbose: print("="*95 + "\n STAGE 3 (Radix-4)\n" + "="*95)
    clk = 0
    for block in range(16): 
        offset = block * 8
        for i in range(2):
            a, b, c, d = stage2_out[offset+i], stage2_out[offset+i+2], stage2_out[offset+i+4], stage2_out[offset+i+6]
            w1, w2, w3 = w(8, i), w(8, 2*i), w(8, 3*i)
            A, B, C, D = radix4_butterfly(a, b, c, d, w1, w2, w3)
            stage3_out[offset+i], stage3_out[offset+i+2], stage3_out[offset+i+4], stage3_out[offset+i+6] = A, B, C, D

            if verbose:
                print(f"[Clk {clk:02d}] Out: A={hex_cpx(A/64, D_BITS, D_FRAC)}, B={hex_cpx(B/64, D_BITS, D_FRAC)}, C={hex_cpx(C/64, D_BITS, D_FRAC)}, D={hex_cpx(D/64, D_BITS, D_FRAC)}")
            clk += 1

    final_output = [0j] * 128
    if verbose: print("="*95 + "\n STAGE 4 (Radix-2 Finisher)\n" + "="*95)
    clk = 0
    for block in range(64):
        offset = block * 2
        A, B = radix2_butterfly(stage3_out[offset], stage3_out[offset + 1])
        final_output[offset], final_output[offset + 1] = A, B
        
        if verbose:
            print(f"[Clk {clk:02d}] Out: A={hex_cpx(A/128, D_BITS, D_FRAC)}, B={hex_cpx(B/128, D_BITS, D_FRAC)}")
        clk += 1

    return final_output

def digit_reverse_128(index):
    stage1 = (index >> 5) & 0x3 
    stage2 = (index >> 3) & 0x3 
    stage3 = (index >> 1) & 0x3 
    stage4 = index & 0x1        
    return (stage4 << 6) | (stage3 << 4) | (stage2 << 2) | stage1

# ==========================================
# Monte Carlo Randomized Testing
# ==========================================
def run_randomized_tests(num_trials=1000):
    print(f"\nRunning {num_trials} randomized Monte Carlo trials...")
    
    max_error_global = 0.0
    sqnr_list = []
    
    for i in range(num_trials):
        # Generate random real signal bounded between -0.99 and 0.99
        random_signal = np.random.uniform(-0.99, 0.99, 128) + 0j
        
        # 1. Hardware Simulation (Silent)
        hw_result_raw = simulate_mdc_pipeline(list(random_signal), verbose=False)
        
        # 2. Re-order results
        hw_result = np.zeros(128, dtype=complex)
        for j in range(128):
            hw_result[digit_reverse_128(j)] = hw_result_raw[j]
            
        # 3. Ideal Floating-Point Math
        ideal_result = np.fft.fft(random_signal)
        
        # 4. Metrics Calculation
        # Absolute Error
        error_array = np.abs(hw_result - ideal_result)
        max_error = np.max(error_array)
        if max_error > max_error_global:
            max_error_global = max_error
            
        # Signal-to-Quantization-Noise Ratio (SQNR) in dB
        signal_power = np.sum(np.abs(ideal_result)**2)
        noise_power = np.sum(error_array**2)
        
        if noise_power > 0:
            sqnr = 10 * np.log10(signal_power / noise_power)
            sqnr_list.append(sqnr)

    avg_sqnr = np.mean(sqnr_list)
    min_sqnr = np.min(sqnr_list)
    
    print("==========================================================")
    print("                 DSP VERIFICATION RESULTS                 ")
    print("==========================================================")
    print(f"Trials Executed       : {num_trials}")
    print(f"Max Absolute Error    : {max_error_global:.6e} (Expected < 1e-10 for unquantized)")
    print(f"Average SQNR          : {avg_sqnr:.2f} dB")
    print(f"Worst-Case SQNR       : {min_sqnr:.2f} dB")
    print("==========================================================")

if __name__ == "__main__":
    # Run the Monte Carlo Bulk Test
    run_randomized_tests(num_trials=500)


Running 500 randomized Monte Carlo trials...
                 DSP VERIFICATION RESULTS                 
Trials Executed       : 500
Max Absolute Error    : 8.881784e-15 (Expected < 1e-10 for unquantized)
Average SQNR          : 310.89 dB
Worst-Case SQNR       : 309.15 dB


In [ ]:
x